<a href="https://colab.research.google.com/github/melaniedaniel7/CFPB-Text-Analysis-Using-NLP-Techniques/blob/main/Text_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Text Cleaning**

### **Objectives:**
1) Import the dataset
2) Clean the text with an example
3) Clean the entire dataset

# **1. Import the dataset**

In [1]:
# Imported the dataset
from google.colab import files

uploaded = files.upload()

Saving complaints-2026-06-17_10_50.csv to complaints-2026-06-17_10_50.csv


In [2]:
# Checked the dataset uploaded correctly
import pandas as pd

df = pd.read_csv("complaints-2026-06-17_10_50.csv")
df.head()
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
       'Company response to consumer', 'Timely response?', 'Complaint ID'],
      dtype='object')

In [3]:
# Checked if any complaint narratives were missing
df.shape
df['Consumer complaint narrative'].isnull().sum()

np.int64(0)

In [4]:
# Dataset only contains complaint narratives now
complaints = df[['Consumer complaint narrative']].copy()
complaints = complaints.dropna()
complaints.shape

(14965, 1)

In [5]:
# Viewed an example complaint
# Realised that I have to clean the redacted text, represented by "XXX's", when doing text cleaning
complaints.head()
print(complaints['Consumer complaint narrative'].iloc[0])

I was told by XXXX XXXX that XXXX XXXX XXXX  gave them information about the status of my account and whether it was paid off or not and how much I owed. I didnt authorize that information to be given to any one.


# **2. Clean the text with an example**

In [6]:
# Import libraries and download NLTK resources for text cleaning
import nltk
# These libraries were not mentioned in phase 1 and were an additional add-on for efficient text preprocessing in phase 2
import re
import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag # Import for POS tagging

# Punkt tokenizer for word tokenization
nltk.download('punkt')
# Stopwords list in multiple languages
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError
nltk.download('wordnet') # Added to resolve LookupError for WordNetLemmatizer
nltk.download('averaged_perceptron_tagger') # Added for POS tagging
nltk.download('averaged_perceptron_tagger_eng') # Added to resolve LookupError for averaged_perceptron_tagger_eng

# Create a set of English stopwords for efficient lookup
stop_words = set(stopwords.words('english'))

# Lemmatizer converts words to their base/root form
lemmatizer = WordNetLemmatizer()

# Function to convert NLTK POS tags to WordNet POS tags
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return nltk.corpus.wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return nltk.corpus.wordnet.VERB
    elif treebank_tag.startswith('N'):
        return nltk.corpus.wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return nltk.corpus.wordnet.ADV
    else:
        return nltk.corpus.wordnet.NOUN # Default to noun if not found

# Function to clean the text by converting text to lowercase and removing punctuation
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        # Remove patterns like 'xxxx' and 'xxxxyear'
        text = re.sub(r'x{2,}', '', text) # This will remove 'xxxx' and 'xxxxyear' will become 'year'
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\s+', ' ', text)
        return text
    else:
        return text

# Function to tokenize the text into individual words
def tokenize_text(text):
    if isinstance(text, str):
        tokens = word_tokenize(text)
        # The previous filter for set(word.lower()) == {'x'} is now less critical
        # as most 'x' sequences are handled by clean_text, but can keep it as a safeguard.
        tokens = [
            word for word in tokens
            if word and not (len(word) == 1 and word.lower() == 'x') # Exclude single 'x' if it appears
        ]
        return tokens
    else:
        return []

# Function to remove stopwords from the tokenized text
def remove_stopwords(tokens):
    if isinstance(tokens, list):
        return [word for word in tokens if word not in stop_words]
    else:
        return tokens

# Function to apply lemmatization to the tokens with POS tagging
def lemmatize_tokens(tokens):
    if isinstance(tokens, list):
        lemmatized_words = []
        for word, tag in pos_tag(tokens):
            wntag = get_wordnet_pos(tag)
            lemmatized_words.append(lemmatizer.lemmatize(word, wntag))
        return lemmatized_words
    else:
        return tokens

# Test text cleaning techniques on a single consumer complaint before applying text cleaning to the entire dataset
# Used the same complaint example as earlier
sample = df['Consumer complaint narrative'].iloc[0]
print("Original:")
print(sample)
# Clean example complaint
cleaned = clean_text(sample)
print("\nCleaned:")
print(cleaned)
# Tokenize example complaint
tokens = tokenize_text(cleaned)
print("\nTokens:")
print(tokens)
# Remove stopwords from example complaint
filtered = remove_stopwords(tokens)
print("\nNO Stpwords:")
print(filtered)
# Lemmatize example complaint
final = lemmatize_tokens(filtered)
print("\nLemmatized:")
print(final)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Original:
I was told by XXXX XXXX that XXXX XXXX XXXX  gave them information about the status of my account and whether it was paid off or not and how much I owed. I didnt authorize that information to be given to any one.

Cleaned:
i was told by that gave them information about the status of my account and whether it was paid off or not and how much i owed i didnt authorize that information to be given to any one

Tokens:
['i', 'was', 'told', 'by', 'that', 'gave', 'them', 'information', 'about', 'the', 'status', 'of', 'my', 'account', 'and', 'whether', 'it', 'was', 'paid', 'off', 'or', 'not', 'and', 'how', 'much', 'i', 'owed', 'i', 'didnt', 'authorize', 'that', 'information', 'to', 'be', 'given', 'to', 'any', 'one']

NO Stpwords:
['told', 'gave', 'information', 'status', 'account', 'whether', 'paid', 'much', 'owed', 'didnt', 'authorize', 'information', 'given', 'one']

Lemmatized:
['told', 'give', 'information', 'status', 'account', 'whether', 'pay', 'much', 'owe', 'didnt', 'authorize

# **3. Clean the entire dataset**

In [7]:
# Apply the functions to the DataFrame
# Clean the text
df['Cleaned_Complaint'] = df['Consumer complaint narrative'].apply(clean_text)
# Tokenize the cleaned text
df['Tokenized_Complaint'] = df['Cleaned_Complaint'].apply(tokenize_text)
# Remove stopwords from the tokenized text
df['No_Stopwords_Complaint'] = df['Tokenized_Complaint'].apply(remove_stopwords)
# Apply lemmatization to the tokenized words
df['Lemmatized_Complaint'] = df['No_Stopwords_Complaint'].apply(lemmatize_tokens)

# Display the original and lemmatized complaint narratives
display(df[['Consumer complaint narrative', 'Lemmatized_Complaint']].head())

,Consumer complaint narrative,Lemmatized_Complaint
0,I was told by XXXX XXXX that XXXX XXXX XXXX g...,"[told, give, information, status, account, whe..."
1,I have been receiving text messages with a hyp...,"[receive, text, message, hyper, link, payment,..."
2,"To Whom It May Concern, I am submitting this c...","[may, concern, submit, complaint, regard, unau..."
3,I am filing a complaint regarding my mortgage ...,"[file, complaint, regard, mortgage, servicer, ..."
4,XXXX XXXX XXXX XXXX XXXX XXXX XXXX NC XXXX XX/...,"[nc, year, professional, finance, co, co, requ..."
